# Open Discourse Analysis

In [ ]:
import importlib, notebook_utils
importlib.reload(notebook_utils)
from notebook_utils import *
conn = get_conn()

## Speeches over time

In [ ]:
df_time = pd.read_sql("""
    SELECT
        EXTRACT(year FROM date)::int AS year,
        COUNT(*) AS speeches
    FROM open_discourse.speeches
    WHERE electoral_term >= %(term_lb)s AND electoral_term <= %(term_ub)s
    GROUP BY year
    ORDER BY year
""", conn, params={"term_lb": TERM_LOWER_BOUNDARY, "term_ub": TERM_UPPER_BOUNDARY})

# WP start years + end for midpoint calculation
wp_boundaries = [1990, 1994, 1998, 2002, 2005, 2009, 2013, 2017, 2021]
wp_names      = ["WP12", "WP13", "WP14", "WP15", "WP16", "WP17", "WP18", "WP19"]
wp_mid = [(wp_boundaries[i] + wp_boundaries[i + 1]) / 2 for i in range(len(wp_names))]

p = (
    ggplot(df_time, aes("year", "speeches"))
    + geom_area(fill=RED, alpha=0.15, colour=RED, size=0.5)
    + geom_vline(xintercept=wp_boundaries[1:-1], colour="#cccccc", size=0.4, linetype="dashed")
    + scale_x_continuous(breaks=wp_boundaries[:-1])
    + scale_y_continuous(labels=lambda l: [f"{v/1000:.0f}k" for v in l])
    + labs(x=None, y="Number of speeches")
    + thesis_theme
)

# WP period labels at midpoints, near the baseline inside the panel
for mid, wp in zip(wp_mid, wp_names):
    p = p + annotate("text", x=mid, y=300, label=wp,
                     ha="center", va="bottom", size=6.5, family="serif", colour="#666666")

p.save(f"{FIGURES}/speeches_per_year.svg", width=TW, height=TW * 0.34, dpi=DPI)
p


In [ ]:
df_time_gender = pd.read_sql("""
    SELECT
        EXTRACT(year FROM s.date)::int AS year,
        p.gender,
        COUNT(*) AS speeches
    FROM open_discourse.speeches s
    JOIN open_discourse.politicians p ON s.politician_id = p.id
    WHERE s.electoral_term >= %(term_lb)s
      AND p.gender IN ('männlich', 'weiblich')
    GROUP BY year, p.gender
    ORDER BY year, p.gender
""", conn, params={"term_lb": TERM_LOWER_BOUNDARY})

df_time_gender["gender_en"] = df_time_gender["gender"].map(GENDER_LABELS)
df_time_gender["gender_en"] = pd.Categorical(
    df_time_gender["gender_en"], categories=["Male", "Female"], ordered=True
)

df_totals = df_time_gender.groupby("year")["speeches"].sum().reset_index(name="total")
df_female = (
    df_time_gender[df_time_gender["gender_en"] == "Female"][["year", "speeches"]]
    .rename(columns={"speeches": "female"})
)
df_share_line = df_totals.merge(df_female, on="year")
df_share_line["female_share"] = df_share_line["female"] / df_share_line["total"]
MAX_TOTAL = int(df_totals["total"].max())
df_share_line["share_scaled"] = df_share_line["female_share"] * MAX_TOTAL
df_share_line["series"] = "Female share"

p = (
    ggplot(df_time_gender, aes("year", "speeches", fill="gender_en"))
    + geom_area(position="stack")
    + geom_line(df_share_line, aes("year", "share_scaled", colour="series"),
                size=0.9, linetype="dashed", inherit_aes=False)
    + scale_fill_manual(values=GENDER_COLORS, name=None)
    + scale_color_manual(values={"Female share": RED}, name=None)
    + scale_x_continuous(breaks=range(1990, 2030, 5))
    + scale_y_continuous(labels=lambda l: [f"{v/1000:.0f}k" for v in l])
    + labs(
        x="Year",
        y="Number of speeches",
        title="Speeches per year by gender ",
    )
    + thesis_theme
)

p.save(f"{FIGURES}/speeches_per_year_gender.svg", width=TW, height=TW * 0.45, dpi=DPI)
p

## Speeches by faction

In [ ]:
df_factions = pd.read_sql(f"""
    SELECT
        {FACTION_MERGE} AS faction,
        COUNT(*) AS speeches
    FROM open_discourse.speeches s
    JOIN open_discourse.factions f ON s.faction_id = f.id
    WHERE f.abbreviation IN ({MAJOR_PARTIES_SQL})
      AND s.electoral_term >= %(term_lb)s
    GROUP BY {FACTION_MERGE}
    ORDER BY speeches DESC
""", conn, params={"term_lb": TERM_LOWER_BOUNDARY})

df_factions["faction"] = relabel_factions(df_factions["faction"] )
df_factions["faction"] = pd.Categorical(
    df_factions["faction"], categories=df_factions["faction"].tolist(), ordered=True
)

p = (
    ggplot(df_factions, aes("faction", "speeches", fill="faction"))
    + bar_geom_col(width=0.7)
    + bar_scale_fill_manual(values=PARTY_COLORS)
    + scale_y_continuous(labels=lambda l: [f"{v/1000:.0f}k" for v in l])
    + labs(
        x=None,
        y="Number of speeches",
        title="Speeches by faction",
    )
    + thesis_theme
    + theme(axis_text_x=element_text(angle=35, ha="right", size=7.5))
)

p.save(f"{FIGURES}/speeches_by_faction.svg", width=TW, height=TW * 0.5, dpi=DPI)
p

## Speeches by electoral term

In [ ]:
df_terms = pd.read_sql("""
    SELECT
        electoral_term,
        COUNT(*) AS speeches,
        MIN(date) AS start_date,
        MAX(date) AS end_date
    FROM open_discourse.speeches
    WHERE electoral_term >= %(term_lb)s
    GROUP BY electoral_term
    ORDER BY electoral_term
""", conn, params={"term_lb": TERM_LOWER_BOUNDARY})

df_terms["label"] = make_wp_labels(df_terms["electoral_term"])

p = (
    ggplot(df_terms, aes("label", "speeches"))
    + bar_geom_col(width=0.75)
    + scale_y_continuous(labels=lambda l: [f"{v/1000:.0f}k" for v in l])
    + labs(
        x="Electoral term",
        y="Number of speeches",
        title="Speeches by term",
    )
    + thesis_theme
    + theme(axis_text_x=element_text(angle=45, ha="right", size=7.5))
)

p.save(f"{FIGURES}/speeches_by_term.svg", width=TW, height=TW * 0.48, dpi=DPI)
p

In [ ]:
df_terms_gender = pd.read_sql("""
    SELECT
        s.electoral_term,
        p.gender,
        SUM(LENGTH(COALESCE(s.speech_content, ''))) AS speeches
    FROM open_discourse.speeches s
    JOIN open_discourse.politicians p ON s.politician_id = p.id
    WHERE s.electoral_term >= %(term_lb)s
      AND s.electoral_term <= %(term_ub)s
      AND p.gender IN ('männlich', 'weiblich')
    GROUP BY s.electoral_term, p.gender
    ORDER BY s.electoral_term, p.gender
""", conn, params={"term_lb": TERM_LOWER_BOUNDARY, "term_ub": TERM_UPPER_BOUNDARY})

df_terms_gender["label"] = make_wp_labels(df_terms_gender["electoral_term"])
df_terms_gender["gender_en"] = pd.Categorical(
    df_terms_gender["gender"].map(GENDER_LABELS),
    categories=["Male", "Female"], ordered=True,
)

p = (
    ggplot(df_terms_gender, aes("label", "speeches", fill="gender_en"))
    + bar_geom_col(width=0.75)
    + bar_scale_fill_manual(values=GENDER_COLORS)
    + scale_y_continuous(labels=lambda l: [f"{v/1e6:.0f}M" for v in l])
    + labs(
        x="Electoral term",
        y="Speech volume (chars)",
        fill=None,
        title="Speech volume by term and gender",
    )
    + thesis_theme
    + theme(
        axis_text_x=element_text(angle=45, ha="right", size=7.5),

        legend_title=element_blank(),
    )
)

p.save(f"{FIGURES}/speeches_by_term_gender.svg", width=TW, height=TW * 0.48, dpi=DPI)
p

In [ ]:
df_terms_party = pd.read_sql(f"""
    SELECT
        s.electoral_term,
        {FACTION_MERGE} AS faction,
        COUNT(*) AS speeches
    FROM open_discourse.speeches s
    JOIN open_discourse.factions f ON s.faction_id = f.id
    WHERE s.electoral_term >= %(term_lb)s
      AND f.abbreviation IN ({MAJOR_PARTIES_SQL})
    GROUP BY s.electoral_term, {FACTION_MERGE}
    ORDER BY s.electoral_term, faction
""", conn, params={"term_lb": TERM_LOWER_BOUNDARY})

df_terms_party["faction"] = relabel_factions(df_terms_party["faction"])
df_terms_party["label"] = make_wp_labels(df_terms_party["electoral_term"])
# Order factions by total speech count (largest at bottom of stack)
faction_order = (
    df_terms_party.groupby("faction")["speeches"].sum()
    .sort_values(ascending=False).index.tolist()
)
df_terms_party["faction"] = pd.Categorical(
    df_terms_party["faction"], categories=faction_order[::-1], ordered=True
)

p = (
    ggplot(df_terms_party, aes("label", "speeches", fill="faction"))
    + bar_geom_col(position="stack", width=0.75)
    + bar_scale_fill_manual(values=PARTY_COLORS)
    + scale_y_continuous(labels=lambda l: [f"{v/1000:.0f}k" for v in l])
    + labs(
        x="Electoral term",
        y="Number of speeches",
        fill=None,
        title="Speeches by term and party",
    )
    + thesis_theme
    + theme(
        axis_text_x=element_text(angle=45, ha="right", size=7.5),

        legend_title=element_blank(),
    )
)

p.save(f"{FIGURES}/speeches_by_term_party.svg", width=TW, height=TW * 0.48, dpi=DPI)
p

## Speaker participation

In [ ]:
# Speech volume per politician (chars spoken, across all post-reunification terms, major parties only)
df_speaker = pd.read_sql(f"""
    SELECT
        s.politician_id,
        {FACTION_MERGE} AS faction,
        p.gender,
        SUM(LENGTH(COALESCE(s.speech_content, ''))) AS total_chars
    FROM open_discourse.speeches s
    JOIN open_discourse.politicians p ON s.politician_id = p.id
    JOIN open_discourse.factions f ON s.faction_id = f.id
    WHERE p.gender IN ('männlich', 'weiblich')
      AND f.abbreviation IN ({MAJOR_PARTIES_SQL})
      AND s.electoral_term >= %(term_lb)s
      AND s.electoral_term <= %(term_ub)s
    GROUP BY s.politician_id, {FACTION_MERGE}, p.gender
    ORDER BY total_chars DESC
""", conn, params={"term_lb": TERM_LOWER_BOUNDARY, "term_ub": TERM_UPPER_BOUNDARY})

df_speaker["faction"] = relabel_factions(df_speaker["faction"])
n_unique = len(df_speaker)
total_vol = df_speaker["total_chars"].sum()
print(f"Unique speakers: {n_unique:,}  |  Total chars: {total_vol/1e9:.2f}B")
print(f"Median chars/speaker: {df_speaker['total_chars'].median():,.0f}  |  Max: {df_speaker['total_chars'].max():,}")
df_speaker.head()

In [ ]:
# Lorenz curve: speech volume concentration across all speakers
df_lorenz = df_speaker.sort_values("total_chars", ascending=False).reset_index(drop=True)
n_spk = len(df_lorenz)
df_lorenz["speaker_pct"] = (df_lorenz.index + 1) / n_spk * 100
df_lorenz["cum_share"] = df_lorenz["total_chars"].cumsum() / df_lorenz["total_chars"].sum() * 100

top10_n = max(1, int(n_spk * 0.1))
top10_share = df_lorenz.iloc[top10_n - 1]["cum_share"]
top1_n = max(1, int(n_spk * 0.01))
top1_share = df_lorenz.iloc[top1_n - 1]["cum_share"]

print(f"Top 10% of speakers → {top10_share:.1f}% of all speech volume")
print(f"Top  1% of speakers → {top1_share:.1f}% of all speech volume")

p = (
    ggplot(df_lorenz, aes("speaker_pct", "cum_share"))
    + geom_abline(intercept=0, slope=1, linetype="dashed", colour="#aaaaaa", size=0.5)
    + geom_line(colour=RED, size=0.8)
    + annotate("text", x=55, y=12,
               label=f"Top 10%: {top10_share:.0f}% of volume\nTop  1%: {top1_share:.0f}% of volume",
               size=7, colour=RED, ha="left")
    + scale_x_continuous(labels=lambda l: [f"{v:.0f}%" for v in l])
    + scale_y_continuous(labels=lambda l: [f"{v:.0f}%" for v in l])
    + labs(
        x="Speaker percentile",
        y="Cumulative speech volume share",
        title="Speech volume concentration: Lorenz curve",
    )
    + thesis_theme
)
p.save(f"{FIGURES}/speaker_lorenz.svg", width=TW, height=TW * 0.5, dpi=DPI)
p

In [ ]:
# Lorenz curves split by gender
df_lorenz_g = []
df_annot_stats = []
for gender_code, gender_label in [("männlich", "Male"), ("weiblich", "Female")]:
    sub = (
        df_speaker[df_speaker["gender"] == gender_code]
        .sort_values("total_chars", ascending=False)
        .reset_index(drop=True)
    )
    n = len(sub)
    sub["speaker_pct"] = (sub.index + 1) / n * 100
    sub["cum_share"] = sub["total_chars"].cumsum() / sub["total_chars"].sum() * 100
    sub["gender_label"] = gender_label
    top10_share = sub.iloc[max(0, int(n * 0.1) - 1)]["cum_share"]
    top1_share  = sub.iloc[max(0, int(n * 0.01) - 1)]["cum_share"]
    print(f"{gender_label} ({n:,} speakers): top 10% → {top10_share:.1f}%, top 1% → {top1_share:.1f}% of within-gender volume")
    df_lorenz_g.append(sub[["speaker_pct", "cum_share", "gender_label"]])
    df_annot_stats.append({"gender_label": gender_label, "top10": top10_share, "top1": top1_share})

df_lorenz_g = pd.concat(df_lorenz_g, ignore_index=True)
df_lorenz_g["gender_label"] = pd.Categorical(
    df_lorenz_g["gender_label"], categories=["Male", "Female"]
)

# Summary statistics as plot annotations
df_annot = pd.DataFrame(df_annot_stats)
df_annot["annotation"] = df_annot.apply(lambda row: f"top 10%: {row['top10']:.1f}%, top 1%: {row['top1']:.1f}%", axis=1)

p = (
    ggplot(df_lorenz_g, aes("speaker_pct", "cum_share"))
    + geom_abline(intercept=0, slope=1, linetype="dashed", colour="#aaaaaa", size=0.5)
    + geom_line(colour="#333333", size=0.8)
    + facet_wrap("~gender_label")
    + coord_equal()
    + geom_text(df_annot, aes(x=5, y=95, label="annotation"), size=7, ha="left", va="top", inherit_aes=False)
    + scale_x_continuous(labels=lambda l: [f"{v:.0f}%" for v in l])
    + scale_y_continuous(labels=lambda l: [f"{v:.0f}%" for v in l])
    + labs(
        x="Speaker percentile (within gender)",
        y="Cumulative speech volume share",
        title="Speech volume concentration by gender",
    )
    + thesis_theme
)
p.save(f"{FIGURES}/speaker_lorenz_by_gender.svg", width=TW, height=TW, dpi=DPI)
p

In [ ]:
# Unique speakers per party
df_n_spk = (
    df_speaker.groupby("faction").size()
    .reset_index(name="n_speakers")
    .sort_values("n_speakers", ascending=False)
)
df_n_spk["faction"] = pd.Categorical(
    df_n_spk["faction"], categories=df_n_spk["faction"].tolist(), ordered=True
)

p = (
    ggplot(df_n_spk, aes("faction", "n_speakers", fill="faction"))
    + bar_geom_col(width=0.7)
    + bar_scale_fill_manual(values=PARTY_COLORS)
    + labs(x=None, y="Unique speakers", title="Unique speakers by party")
    + thesis_theme
    + theme(axis_text_x=element_text(angle=35, ha="right", size=7.5))
)
p.save(f"{FIGURES}/speakers_per_party.svg", width=TW * 0.6, height=TW * 0.45, dpi=DPI)
p

In [ ]:
from plotnine import geom_boxplot

# Speech volume distribution by party (log scale)
median_order = (
    df_speaker.groupby("faction")["total_chars"].median()
    .sort_values(ascending=False).index.tolist()
)
df_speaker["faction_ord"] = pd.Categorical(
    df_speaker["faction"], categories=median_order, ordered=True
)
df_speaker["log10_chars"] = np.log10(df_speaker["total_chars"].clip(lower=1))

p = (
    ggplot(df_speaker, aes("faction_ord", "log10_chars", fill="faction_ord"))
    + geom_boxplot(outlier_alpha=0.25, outlier_size=0.6, width=0.6)
    + scale_fill_manual(values=PARTY_COLORS)
    + scale_y_continuous(
        breaks=[2, 3, 4, 5, 6, 7],
        labels=["100", "1k", "10k", "100k", "1M", "10M"],
    )
    + labs(
        x=None,
        y="Speech volume (chars)",
        title="Speech volume distribution by party (log scale)",
    )
    + thesis_theme
    + theme(axis_text_x=element_text(angle=35, ha="right", size=7.5))
)
p.save(f"{FIGURES}/speaker_distribution_by_party.svg", width=TW, height=TW * 0.5, dpi=DPI)
p

## Gender

In [ ]:
# Female share of speeches per electoral term (reuses df_terms_gender from cell above)
df_pivot = df_terms_gender.pivot(
    index="electoral_term", columns="gender", values="speeches"
).fillna(0)
df_pivot["female_share"] = df_pivot["weiblich"] / (df_pivot["männlich"] + df_pivot["weiblich"])
df_share = df_pivot.reset_index()[["electoral_term", "female_share"]]
df_share["label"] = make_wp_labels(df_share["electoral_term"])

p = (
    ggplot(df_share, aes("label", "female_share"))
    + bar_geom_col(fill=RED, alpha=0.85, width=0.75)
    + scale_y_continuous(labels=lambda l: [f"{v*100:.0f}%" for v in l])
    + labs(
        x="Electoral term",
        y="Female share of speeches",
        title="Female speech share by term",
    )
    + thesis_theme
    + theme(axis_text_x=element_text(angle=45, ha="right", size=7.5))
)

p.save(f"{FIGURES}/female_share_by_term.svg", width=TW, height=TW * 0.48, dpi=DPI)
p

In [ ]:
# Female share of speech volume by faction
df_gender_faction = pd.read_sql(f"""
    SELECT
        {FACTION_MERGE} AS faction,
        p.gender,
        SUM(LENGTH(COALESCE(s.speech_content, ''))) AS speeches
    FROM open_discourse.speeches s
    JOIN open_discourse.politicians p ON s.politician_id = p.id
    JOIN open_discourse.factions f ON s.faction_id = f.id
    WHERE p.gender IN ('männlich', 'weiblich')
      AND f.abbreviation IN ({MAJOR_PARTIES_SQL})
      AND s.electoral_term >= %(term_lb)s
      AND s.electoral_term <= %(term_ub)s
    GROUP BY {FACTION_MERGE}, p.gender
""", conn, params={"term_lb": TERM_LOWER_BOUNDARY, "term_ub": TERM_UPPER_BOUNDARY})

df_gender_faction["faction"] = relabel_factions(df_gender_faction["faction"])
df_fp = df_gender_faction.pivot(index="faction", columns="gender", values="speeches").fillna(0)
df_fp["female_share"] = df_fp["weiblich"] / (df_fp["männlich"] + df_fp["weiblich"])
df_fp = df_fp.reset_index().sort_values("female_share", ascending=False)
df_fp["faction"] = pd.Categorical(df_fp["faction"], categories=df_fp["faction"].tolist(), ordered=True)

p = (
    ggplot(df_fp, aes("faction", "female_share", fill="faction"))
    + bar_geom_col(width=0.65)
    + bar_scale_fill_manual(values=PARTY_COLORS)
    + scale_y_continuous(labels=lambda l: [f"{v*100:.0f}%" for v in l], limits=(0, 0.55))
    + labs(
        x=None,
        y="Female share of speech volume",
        title="Female speech volume share by faction",
    )
    + thesis_theme
)

p.save(f"{FIGURES}/female_share_by_faction.svg", width=TW, height=TW * 0.45, dpi=DPI)
p

## Female seat share (Bundestag Stammdaten)

In [ ]:
df_seats = parse_mdb_seats()
print(f"Seat records (WP {TERM_LOWER_BOUNDARY}+): {len(df_seats):,}")
print(df_seats.groupby("gender").size().to_dict())
df_seats.head(3)

In [ ]:
df_seat_share_area = (
    df_seats[df_seats["gender"].isin(["männlich", "weiblich"]) & (df_seats["wp"] <= TERM_UPPER_BOUNDARY)]
    .groupby(["wp", "gender"])
    .size()
    .unstack(fill_value=0)
    .assign(female_share=lambda d: d["weiblich"] / (d["männlich"] + d["weiblich"]))
    .reset_index()[["wp", "female_share"]]
 )
df_seat_share_area["label"] = make_wp_labels(df_seat_share_area["wp"])

p = (
    ggplot(df_seat_share_area, aes("label", "female_share"))
    + bar_geom_bar(stat='identity', fill=GRAY, alpha=0.7)
    + scale_y_continuous(
        labels=lambda l: [f"{v*100:.0f}%" for v in l],
        limits=(0, None),
    )
    + labs(
        x="Electoral term",
        y="Female seat share",
        #title="Bundestag seats filled with female MPs by electoral term ",
    )
    + thesis_theme
    + theme(axis_text_x=element_text(angle=45, ha="right", size=7.5))
)

p.save(f"{FIGURES}/female_seat_share.svg", width=TW, height=TW * 0.45, dpi=DPI)
p

In [ ]:
# Stacked bar: seat share by gender per electoral term
df_seat_gender_stack = (
    df_seats[df_seats["gender"].isin(["männlich", "weiblich"]) & (df_seats["wp"] <= TERM_UPPER_BOUNDARY)]
    .groupby(["wp", "gender"])
    .size()
    .reset_index(name="count")
)
df_seat_gender_stack["share"] = (
    df_seat_gender_stack["count"]
    / df_seat_gender_stack.groupby("wp")["count"].transform("sum")
)
df_seat_gender_stack["label"] = make_wp_labels(df_seat_gender_stack["wp"])
df_seat_gender_stack["gender_en"] = df_seat_gender_stack["gender"].map(GENDER_LABELS)
df_seat_gender_stack["gender_en"] = pd.Categorical(
    df_seat_gender_stack["gender_en"], categories=["Male", "Female"], ordered=True
)

p = (
    ggplot(df_seat_gender_stack, aes("label", "share", fill="gender_en"))
    + bar_geom_bar(stat="identity", width=0.7)
    + bar_scale_fill_manual(values={"Male": "#444444", "Female": "#bbbbbb"})
    + scale_y_continuous(labels=lambda l: [f"{v*100:.0f}%" for v in l])
    + labs(x="Electoral term", y="Seat share", fill=None)
    + thesis_theme
    + theme(axis_text_x=element_text(angle=45, ha="right", size=7.5))
)

p.save(f"{FIGURES}/seat_share_by_gender.svg", width=TW, height=TW * 0.45, dpi=DPI)
p


In [ ]:
# Female seat share by electoral term and party
df_seat_share_by_term_party = (
    df_seats[df_seats["gender"].isin(["männlich", "weiblich"]) & (df_seats["wp"] <= TERM_UPPER_BOUNDARY)]
    .groupby(["wp", "faction", "gender"])
    .size()
    .unstack(fill_value=0)
    .assign(female_share=lambda d: d["weiblich"] / (d["männlich"] + d["weiblich"]))
    .reset_index()[["wp", "faction", "female_share"]]
)
df_seat_share_by_term_party["label"] = make_wp_labels(df_seat_share_by_term_party["wp"])
df_seat_share_by_term_party["faction"] = relabel_factions(df_seat_share_by_term_party["faction"])

# Filter out "Other" faction
df_seat_share_by_term_party = df_seat_share_by_term_party[df_seat_share_by_term_party["faction"] != "Other"]

# Order factions by total female share (largest first)
faction_order = (
    df_seat_share_by_term_party.groupby("faction")["female_share"].mean()
    .sort_values(ascending=False).index.tolist()
)
df_seat_share_by_term_party["faction"] = pd.Categorical(
    df_seat_share_by_term_party["faction"], categories=faction_order, ordered=True
)

p = (
    ggplot(df_seat_share_by_term_party, aes("label", "female_share", color="faction"))
    + geom_line(aes(group="faction"), size=0.6)
    + geom_point(aes(group="faction"), size=1.2)
    + geom_line(data=df_seat_share_area, mapping=aes("label", "female_share"), color=GRAY, size=0.6, linetype="solid", group=1, inherit_aes=False)
    + geom_point(data=df_seat_share_area, mapping=aes("label", "female_share"), color=GRAY, size=1.2, inherit_aes=False)
    + scale_color_manual(values=PARTY_COLORS)
    + scale_y_continuous(
        labels=lambda l: [f"{v*100:.0f}%" for v in l],
        limits=(None, 0.6),
    )
    + labs(
        x="Electoral term",
        y="Female seat share",
        color="Faction"
    )
    + thesis_theme
    + theme(axis_text_x=element_text(angle=45, ha="right", size=7.5))
)

p.save(f"{FIGURES}/female_seat_share_by_term_party.svg", width=TW * 1.2, height=TW * 0.68, dpi=DPI)
p


In [ ]:
# Average female seat share by faction (collapsed across terms)
df_seat_share_by_faction = (
    df_seat_share_by_term_party
    .groupby("faction")["female_share"]
    .mean()
    .reset_index()
    .sort_values("female_share", ascending=False)
)
df_seat_share_by_faction["faction"] = pd.Categorical(
    df_seat_share_by_faction["faction"],
    categories=df_seat_share_by_faction["faction"].tolist(),
    ordered=True
)

p = (
    ggplot(df_seat_share_by_faction, aes("faction", "female_share", fill="faction"))
    + bar_geom_col(width=0.65)
    + bar_scale_fill_manual(values=PARTY_COLORS)
    + scale_y_continuous(labels=lambda l: [f"{v*100:.0f}%" for v in l], limits=(0, 0.6))
    + labs(x=None, y="Female seat share (avg. across terms)")
    + thesis_theme
    + theme(legend_position="none")
)

p.save(f"{FIGURES}/female_seat_share_by_faction.svg", width=TW, height=TW * 0.45, dpi=DPI)
p


In [ ]:
# Female seat share per electoral term vs. female speech share
df_seat_share = (
    df_seats[
        df_seats["gender"].isin(["männlich", "weiblich"]) &
        (df_seats["wp"] <= TERM_UPPER_BOUNDARY)
    ]
    .groupby(["wp", "gender"])
    .size()
    .unstack(fill_value=0)
    .assign(female_share=lambda d: d["weiblich"] / (d["männlich"] + d["weiblich"]))
    .reset_index()[["wp", "female_share"]]
    .rename(columns={"wp": "electoral_term", "female_share": "seats"})
)

# Reuse df_share from the speeches cell (female share of speeches per term)
df_ctrl = df_share[["electoral_term", "female_share"]].rename(
    columns={"female_share": "speeches"}
)
df_ctrl = df_ctrl.merge(df_seat_share, on="electoral_term")
df_ctrl_long = df_ctrl.melt(id_vars="electoral_term", var_name="series", value_name="share")
df_ctrl_long["label"] = make_wp_labels(df_ctrl_long["electoral_term"])

SERIES_COLORS = {"speeches": RED, "seats": "#4a4a4a"}

p = (
    ggplot(df_ctrl_long, aes("label", "share", colour="series", group="series"))
    + geom_line(size=0.8)
    + geom_point(size=1.5)
    + scale_color_manual(
        values=SERIES_COLORS,
        labels={"speeches": "Speech share", "seats": "Seat share"},
    )
    + scale_y_continuous(
        labels=lambda l: [f"{v*100:.0f}%" for v in l],
    )
    + labs(
        x="Electoral term",
        y="Female share",
        colour=None,
        title="Female seat share vs. speech share by electoral term",
    )
    + thesis_theme
    + theme(
        axis_text_x=element_text(angle=45, ha="right", size=7.5),

        legend_title=element_blank(),
    )
)

p.save(f"{FIGURES}/female_seat_vs_speech_share.svg", width=TW, height=TW * 0.5, dpi=DPI)
p

In [ ]:
# Delta: speech share minus seat share per electoral term
df_delta = df_ctrl.copy()
df_delta["delta"] = df_delta["speeches"] - df_delta["seats"]
df_delta["label"] = make_wp_labels(df_delta["electoral_term"])
df_delta["direction"] = df_delta["delta"].map(lambda x: "above" if x >= 0 else "below")

p_delta = (
    ggplot(df_delta, aes("label", "delta", fill="direction"))
    + bar_geom_col(alpha=0.85, width=0.75)
    + geom_hline(yintercept=0, size=0.5, colour="#555555", linetype="dashed")
    + bar_scale_fill_manual(values={"above": RED, "below": "#666666"}, guide=None)
    + scale_y_continuous(labels=lambda l: [f"{v*100:+.0f}pp" for v in l])
    + labs(
        x="Electoral term (Wahlperiode)",
        y="Speech share − Seat share",
        title="Female speech share minus seat share by electoral term",
    )
    + thesis_theme
    + theme(axis_text_x=element_text(angle=45, ha="right", size=7.5))
)

p_delta.save(f"{FIGURES}/female_share_delta_by_term.svg", width=TW, height=TW * 0.45, dpi=DPI)
p_delta

In [ ]:
# Delta by party × electoral term: female speech volume share minus female seat share
df_sft = pd.read_sql(f"""
    SELECT
        s.electoral_term,
        {FACTION_MERGE} AS faction,
        p.gender,
        SUM(LENGTH(COALESCE(s.speech_content, ''))) AS speeches
    FROM open_discourse.speeches s
    JOIN open_discourse.politicians p ON s.politician_id = p.id
    JOIN open_discourse.factions f ON s.faction_id = f.id
    WHERE p.gender IN ('männlich', 'weiblich')
      AND f.abbreviation IN ({MAJOR_PARTIES_SQL})
      AND s.electoral_term >= %(term_lb)s
      AND s.electoral_term <= %(term_ub)s
    GROUP BY s.electoral_term, {FACTION_MERGE}, p.gender
""", conn, params={"term_lb": TERM_LOWER_BOUNDARY, "term_ub": TERM_UPPER_BOUNDARY})

df_sft["faction"] = relabel_factions(df_sft["faction"])

df_speech_share = (
    df_sft.pivot_table(
        index=["electoral_term", "faction"], columns="gender", values="speeches", fill_value=0
    )
    .assign(speech_female_share=lambda d: d["weiblich"] / (d["männlich"] + d["weiblich"]))
    .reset_index()[["electoral_term", "faction", "speech_female_share"]]
)

# Seat share by faction × term from the MDB Stammdaten (df_seats already loaded above)
df_seat_ft = (
    df_seats[
        df_seats["gender"].isin(["männlich", "weiblich"]) &
        df_seats["faction"].isin(PARTY_COLORS.keys()) &
        (df_seats["wp"] <= TERM_UPPER_BOUNDARY)
    ]
    .groupby(["wp", "faction", "gender"])
    .size().unstack(fill_value=0)
    .assign(seat_female_share=lambda d: d["weiblich"] / (d["männlich"] + d["weiblich"]))
    .reset_index()[["wp", "faction", "seat_female_share"]]
    .rename(columns={"wp": "electoral_term"})
)

df_delta_party = (
    df_speech_share.merge(df_seat_ft, on=["electoral_term", "faction"], how="inner")
    .assign(delta=lambda d: d["speech_female_share"] - d["seat_female_share"])
)
df_delta_party["label"] = make_wp_labels(df_delta_party["electoral_term"])

faction_order = (
    df_sft.groupby("faction")["speeches"].sum()
    .sort_values(ascending=False).index.tolist()
)
df_delta_party["faction"] = pd.Categorical(
    df_delta_party["faction"], categories=faction_order, ordered=True
)

p = (
    ggplot(df_delta_party, aes("label", "delta", colour="faction", group="faction"))
    + geom_hline(yintercept=0, linetype="dashed", colour="#aaaaaa", size=0.4)
    + geom_line(size=0.75)
    + geom_point(size=1.4)
    + scale_color_manual(values=PARTY_COLORS, name=None)
    + scale_y_continuous(labels=lambda l: [f"{v*100:+.0f}pp" for v in l])
    + labs(
        x="Electoral term",
        y="Female speech volume share − Female seat share",
        title="Gender gap delta (speech volume) by party and electoral term",
    )
    + thesis_theme
    + theme(
        axis_text_x=element_text(angle=45, ha="right", size=7.5),

    )
)
p.save(f"{FIGURES}/female_share_delta_by_party_term.svg", width=TW, height=TW * 0.5, dpi=DPI)
p

In [ ]:
df_table = df_delta[["label", "seats", "speeches", "delta"]].copy()
df_table.columns = ["Electoral term", "Seat share", "Speech share", "Delta (speech − seat)"]
df_table = df_table.set_index("Electoral term")

df_table.style.format({
    "Seat share": "{:.1%}",
    "Speech share": "{:.1%}",
    "Delta (speech − seat)": "{:+.1%}",
})

### Voice per seat (chars / n_seats)

In [ ]:
# Voice per seat: chars spoken / number of seats held, by gender × term.
# Both sources cover the same scope

# Seat counts by gender × term (MDB Stammdaten XML)
df_seats_gt = (
    df_seats[
        df_seats["gender"].isin(["männlich", "weiblich"]) &
        (df_seats["wp"] <= TERM_UPPER_BOUNDARY)
    ]
    .groupby(["wp", "gender"])
    .size()
    .reset_index(name="n_seats")
    .rename(columns={"wp": "electoral_term"})
)
print("=== Step 1: seat counts (MDB Stammdaten XML) ===")
print(f"Rows: {len(df_seats_gt)}  |  Terms: {sorted(df_seats_gt['electoral_term'].unique())}")
print(df_seats_gt.pivot(index="electoral_term", columns="gender", values="n_seats")
      .rename_axis(None, axis=1).to_string())

# Char volume by gender × term (Open Discourse speeches DB)
df_chars_gt = (
    df_terms_gender[["electoral_term", "gender", "speeches"]]
    .rename(columns={"speeches": "total_chars"})
)
print("\n=== Step 2: char volume (speeches DB, in millions) ===")
print(f"Rows: {len(df_chars_gt)}  |  Terms: {sorted(df_chars_gt['electoral_term'].unique())}")
pivot_chars = df_chars_gt.pivot(index="electoral_term", columns="gender", values="total_chars")
print((pivot_chars / 1e6).round(1).rename_axis(None, axis=1).to_string())

# Term alignment check
terms_seats = set(df_seats_gt["electoral_term"].unique())
terms_chars = set(df_chars_gt["electoral_term"].unique())
only_in_seats = sorted(terms_seats - terms_chars)
only_in_chars = sorted(terms_chars - terms_seats)
EXPECTED_XML_ONLY = {21}
unexpected_xml_only = [t for t in only_in_seats if t not in EXPECTED_XML_ONLY]
print("\n=== Step 3: term alignment ===")
print(f"XML only (expected — corpus not yet updated): {[t for t in only_in_seats if t in EXPECTED_XML_ONLY] or 'none'}")
print(f"XML only (UNEXPECTED): {unexpected_xml_only or 'none'}")
print(f"DB only  (unexpected): {sorted(only_in_chars) or 'none'}")
print(f"Matched terms: {sorted(terms_seats & terms_chars)}")

# Merge with outer join + indicator
df_voice = df_seats_gt.merge(
    df_chars_gt, on=["electoral_term", "gender"], how="outer", indicator=True
)
merge_summary = df_voice["_merge"].value_counts().to_dict()
print("\n=== Step 4: merge diagnostics ===")
for status, n in sorted(merge_summary.items()):
    expected = status == "both" or (status == "left_only" and n == len(EXPECTED_XML_ONLY) * 2)
    flag = "" if expected else " ← WARNING"
    print(f"  {status}: {n} rows{flag}")

unexpected_unmatched = df_voice[
    (df_voice["_merge"] != "both") &
    (~df_voice["electoral_term"].isin(EXPECTED_XML_ONLY))
]
if len(unexpected_unmatched):
    print("UNEXPECTED unmatched rows:")
    print(unexpected_unmatched.to_string())
else:
    print("All unexpected unmatched rows: none.")

df_voice = df_voice[df_voice["_merge"] == "both"].drop(columns="_merge")

# Voice metric
df_voice["chars_per_seat"] = df_voice["total_chars"] / df_voice["n_seats"]

print("\n=== Step 5: chars per seat (thousands) ===")
pivot_cps = df_voice.pivot(index="electoral_term", columns="gender", values="chars_per_seat")
print((pivot_cps / 1e3).round(1).rename_axis(None, axis=1).to_string())

df_ratio = (
    pivot_cps
    .assign(voice_ratio=lambda d: d["weiblich"] / d["männlich"])
    .reset_index()
)
df_ratio["label"] = make_wp_labels(df_ratio["electoral_term"])

print("\n=== Voice ratio (female / male chars per seat) ===")
summary = df_ratio[["electoral_term", "weiblich", "männlich", "voice_ratio"]].copy()
summary["weiblich"] = (summary["weiblich"] / 1e3).round(1)
summary["männlich"] = (summary["männlich"] / 1e3).round(1)
summary["voice_ratio"] = summary["voice_ratio"].round(3)
print(summary.rename(columns={
    "weiblich": "F (k chars/seat)", "männlich": "M (k chars/seat)", "voice_ratio": "F/M ratio"
}).to_string(index=False))

# Flag incomplete terms
# A term is suspect if its total char volume is less than 50% of the median.
median_vol = df_voice.groupby("electoral_term")["total_chars"].sum().median()
term_vols = df_voice.groupby("electoral_term")["total_chars"].sum()
incomplete_terms = term_vols[term_vols < 0.5 * median_vol].index.tolist()
print(f"\n=== Step 6: low-volume terms (< 50% of median {median_vol/1e6:.0f}M chars) ===")
print(f"Flagged as potentially incomplete: {sorted(incomplete_terms)}")
print("Interpret ratio for these terms with caution — corpus likely not fully processed.")
df_ratio["incomplete"] = df_ratio["electoral_term"].isin(incomplete_terms)

In [ ]:
# Absolute chars/seat by gender
df_voice_long = df_voice[["electoral_term", "gender", "chars_per_seat"]].copy()
df_voice_long["gender_en"] = pd.Categorical(
    df_voice_long["gender"].map(GENDER_LABELS),
    categories=["Male", "Female"], ordered=True,
)
df_voice_long["label"] = make_wp_labels(df_voice_long["electoral_term"])
df_voice_long["chars_per_seat_k"] = df_voice_long["chars_per_seat"] / 1000

p1 = (
    ggplot(df_voice_long, aes("label", "chars_per_seat_k", colour="gender_en", group="gender_en"))
    + geom_line(size=0.8)
    + geom_point(size=1.5)
    + scale_color_manual(values=GENDER_COLORS, name=None)
    + scale_y_continuous(labels=lambda l: [f"{v:.0f}k" for v in l])
    + labs(
        x="Electoral term",
        y="Speech volume per seat (1 000 chars)",
        title="Voice per seat by gender",
    )
    + thesis_theme
    + theme(
        axis_text_x=element_text(angle=45, ha="right", size=7.5),

        legend_title=element_blank(),
    )
)
p1.save(f"{FIGURES}/voice_per_seat_by_gender.svg", width=TW, height=TW * 0.5, dpi=DPI)
p1

In [ ]:
# Voice ratio (female / male chars per seat) over time
df_ratio_complete = df_ratio[~df_ratio["incomplete"]]
df_ratio_incomplete = df_ratio[df_ratio["incomplete"]]

p2 = (
    ggplot(df_ratio_complete, aes("label", "voice_ratio", group=1))
    + geom_hline(yintercept=1, linetype="dashed", colour="#aaaaaa", size=0.5)
    + geom_line(colour=RED, size=0.8)
    + geom_point(colour=RED, size=1.8)
    + scale_y_continuous(labels=lambda l: [f"{v:.2f}" for v in l])
    + labs(
        x="Electoral term",
        y="Female / male chars per seat",
        title="Voice ratio: female chars per seat relative to male",
    )
    + thesis_theme
    + theme(axis_text_x=element_text(angle=45, ha="right", size=7.5))
)

if len(df_ratio_incomplete):
    p2 = (
        p2
        + geom_point(data=df_ratio_incomplete, colour=RED, size=1.8, fill="white", stroke=0.8)
        + annotate(
            "text",
            x=df_ratio_incomplete["label"].iloc[0],
            y=df_ratio_incomplete["voice_ratio"].iloc[0] + 0.03,
            label="corpus\nincomplete", size=6, colour="#888888", ha="center",
        )
    )

p2.save(f"{FIGURES}/voice_ratio_by_term.svg", width=TW, height=TW * 0.5, dpi=DPI)
p2

- 0 = parity
- Negative: Women speak less than men per seat
- Positive: Women speak more than men per seat

### Decomposing the speech share gap

In [ ]:
# Diagnostic: factions in corpus + speech length distribution
df_factions_all = pd.read_sql("""
    SELECT id, abbreviation, full_name FROM open_discourse.factions ORDER BY abbreviation
""", conn)
print("=== Factions ===")
print(df_factions_all.to_string(index=False))

df_len_dist = pd.read_sql("""
    SELECT
        CASE
            WHEN LENGTH(COALESCE(speech_content,'')) = 0    THEN '0 (empty)'
            WHEN LENGTH(COALESCE(speech_content,'')) < 200  THEN '1–199 chars'
            WHEN LENGTH(COALESCE(speech_content,'')) < 500  THEN '200–499 chars'
            WHEN LENGTH(COALESCE(speech_content,'')) < 2000 THEN '500–1999 chars'
            ELSE '2000+ chars'
        END AS bucket,
        COUNT(*) AS n_speeches,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM open_discourse.speeches
    WHERE electoral_term >= %(term_lb)s
    GROUP BY bucket ORDER BY MIN(LENGTH(COALESCE(speech_content,'')))
""", conn, params={"term_lb": TERM_LOWER_BOUNDARY})
print("\n=== Speech length distribution ===")
print(df_len_dist.to_string(index=False))

# Identify Bundestagspräsident/in politician IDs
df_presidents = pd.read_sql("""
    SELECT id, first_name, last_name, gender
    FROM open_discourse.politicians
    WHERE last_name IN ('Süssmuth', 'Thierse', 'Lammert', 'Schäuble', 'Bas')
      AND first_name IN ('Rita', 'Wolfgang', 'Norbert', 'Bärbel')
""", conn)
print("\n=== Bundestagspräsident/in candidates ===")
print(df_presidents.to_string(index=False))

In [ ]:
# Count-based female share (old metric, for comparison)
df_by_count = pd.read_sql("""
    SELECT s.electoral_term, p.gender, COUNT(*) AS n
    FROM open_discourse.speeches s
    JOIN open_discourse.politicians p ON s.politician_id = p.id
    WHERE p.gender IN ('männlich', 'weiblich')
      AND s.electoral_term >= %(term_lb)s
      AND s.electoral_term <= %(term_ub)s
    GROUP BY s.electoral_term, p.gender
""", conn, params={"term_lb": TERM_LOWER_BOUNDARY, "term_ub": TERM_UPPER_BOUNDARY})

df_count_share = (
    df_by_count
    .pivot_table(index="electoral_term", columns="gender", values="n", fill_value=0)
    .assign(share=lambda d: d["weiblich"] / (d["männlich"] + d["weiblich"]))
    .reset_index()[["electoral_term", "share"]]
    .assign(series="Speech count share")
)

# Char-based female share (new metric)
df_char_share = (
    df_terms_gender
    .pivot_table(index="electoral_term", columns="gender", values="speeches", fill_value=0)
    .assign(share=lambda d: d["weiblich"] / (d["männlich"] + d["weiblich"]))
    .reset_index()[["electoral_term", "share"]]
    .assign(series="Speech volume share (chars)")
)

# Token-based female share
df_by_tokens = pd.read_sql(r"""
    SELECT s.electoral_term, p.gender,
           SUM(array_length(regexp_split_to_array(s.speech_content, '\s+'), 1)) AS tokens
    FROM open_discourse.speeches s
    JOIN open_discourse.politicians p ON s.politician_id = p.id
    WHERE p.gender IN ('männlich', 'weiblich')
      AND s.electoral_term >= %(term_lb)s
      AND s.electoral_term <= %(term_ub)s
    GROUP BY s.electoral_term, p.gender
""", conn, params={"term_lb": TERM_LOWER_BOUNDARY, "term_ub": TERM_UPPER_BOUNDARY})

df_token_share = (
    df_by_tokens
    .pivot_table(index="electoral_term", columns="gender", values="tokens", fill_value=0)
    .assign(share=lambda d: d["weiblich"] / (d["männlich"] + d["weiblich"]))
    .reset_index()[["electoral_term", "share"]]
    .assign(series="Speech volume share (tokens)")
)

# Combine with seat share as reference
df_seat_ref = df_seat_share.rename(columns={"seats": "share"}).assign(series="Seat share")

df_decomp = pd.concat(
    [df_count_share, df_char_share, df_token_share, df_seat_ref], ignore_index=True
)
df_decomp["label"] = make_wp_labels(df_decomp["electoral_term"])

print("Count-based vs char-based female share (WP12–WP19):")
comp = df_count_share.merge(df_char_share, on="electoral_term", suffixes=("_count","_chars"))
comp["diff_pp"] = (comp["share_chars"] - comp["share_count"]) * 100
comp[["electoral_term","share_count","share_chars","diff_pp"]]
# Max absolute difference (pp) between chars and tokens volume share
_chars = df_char_share.set_index("electoral_term")["share"]
_tokens = df_token_share.set_index("electoral_term")["share"]
chars_tokens_max_diff_pp = float((_chars - _tokens).abs().max() * 100)
print(f"Max chars-tokens diff: {chars_tokens_max_diff_pp:.2f} pp")


In [ ]:
DECOMP_COLORS = {
    "Seat share":               "#cccccc",
    "Speech count share":       "#888888",
    "Speech volume share":      "#333333",
}
DECOMP_LINES = {
    "Seat share":               "dotted",
    "Speech count share":       "dashed",
    "Speech volume share":      "solid",
}

df_decomp_plot = df_decomp[
    df_decomp["series"] != "Speech volume share (tokens)"
].copy()
df_decomp_plot["series"] = df_decomp_plot["series"].replace(
    {"Speech volume share (chars)": "Speech volume share"}
)
series_order = ["Seat share", "Speech count share", "Speech volume share"]
df_decomp_plot["series"] = pd.Categorical(
    df_decomp_plot["series"], categories=series_order, ordered=True
)

p = (
    ggplot(df_decomp_plot, aes("label", "share", colour="series", group="series", linetype="series"))
    + geom_line(size=0.75)
    + geom_point(size=1.4)
    + scale_color_manual(values=DECOMP_COLORS, name=None)
    + scale_linetype_manual(values=DECOMP_LINES, name=None)
    + scale_y_continuous(labels=lambda l: [f"{v*100:.0f}%" for v in l])
    + labs(x="Electoral term", y="Female share")
    + thesis_theme
    + theme(axis_text_x=element_text(angle=45, ha="right", size=7.5))
)
p.save(f"{FIGURES}/speech_share_decomposed.svg", width=TW, height=TW * 0.6, dpi=DPI)
p


In [ ]:
os.makedirs(TABLES_DIR, exist_ok=True)

df_decomp_tbl = df_decomp[df_decomp["series"] != "Speech volume share (tokens)"].copy()
df_decomp_tbl["series"] = df_decomp_tbl["series"].replace(
    {"Speech volume share (chars)": "Speech volume share"}
)
df_tbl = (
    df_decomp_tbl
    .assign(share_fmt=lambda d: d["share"].map(lambda x: f"{x*100:.1f}%"))
    .pivot_table(index=["electoral_term", "label"], columns="series",
                 values="share_fmt", aggfunc="first")
    .reset_index()
)
df_tbl.columns.name = None
series_cols = ["Seat share", "Speech count share", "Speech volume share"]
df_tbl = df_tbl.rename(columns={"label": "Electoral term"})[["Electoral term"] + series_cols]

def _esc(v):
    s = str(v)
    if s in ("", "nan", "None"):
        return "[\u2014]"
    s = s.replace("%", "\\%").replace("<", "\\<").replace("[", "\\[").replace("]", "\\]")
    return "[" + s + "]"

col_names = list(df_tbl.columns)
n = len(col_names)
col_align = "left, " + ", ".join(["right"] * (n - 1))
header = "  table.header(" + ", ".join(f"[*{c}*]" for c in col_names) + "),"
rows = []
for _, row in df_tbl.iterrows():
    cells = ", ".join(_esc(row[c]) for c in col_names)
    rows.append(f"  {cells},")

lines = [
    "#table(",
    f"  columns: {n},",
    "  stroke: none,",
    "  inset: (x: 5pt, y: 3pt),",
    f"  align: ({col_align}),",
    "  table.hline(stroke: 1.5pt),",
    header,
    "  table.hline(y: 1, stroke: .75pt),",
] + rows + [
    "  table.hline(stroke: 1.5pt),",
    ")",
]

with open(os.path.join(TABLES_DIR, "tbl_speech_share_decomp.typ"), "w") as f:
    f.write("\n".join(lines) + "\n")


## Interjections & Interruptions

### Interjection rates by type and speaker gender


In [ ]:
# Interjection rates received per 100k chars of speech, by speaker gender
# Char-based normalisation avoids inflation from many short speeches.
df_inj = pd.read_sql("""
    WITH speech_vol AS (
        SELECT p.gender, SUM(LENGTH(COALESCE(s.speech_content, ''))) AS total_chars
        FROM open_discourse.speeches s
        JOIN open_discourse.politicians p ON s.politician_id = p.id
        WHERE p.gender IN ('männlich', 'weiblich')
          AND s.electoral_term >= %(term_lb)s
          AND s.electoral_term <= %(term_ub)s
        GROUP BY p.gender
    ),
    inj_counts AS (
        SELECT p.gender, ce.type, COUNT(*) AS n
        FROM open_discourse.contributions_extended ce
        JOIN open_discourse.speeches s ON ce.speech_id = s.id
        JOIN open_discourse.politicians p ON s.politician_id = p.id
        WHERE p.gender IN ('männlich', 'weiblich')
          AND ce.type IN ('Beifall', 'Zuruf', 'Widerspruch', 'Lachen', 'Heiterkeit')
          AND s.electoral_term >= %(term_lb)s
          AND s.electoral_term <= %(term_ub)s
        GROUP BY p.gender, ce.type
    )
    SELECT ic.gender, ic.type,
           ic.n::numeric / sv.total_chars * 100000 AS per_100k_chars
    FROM inj_counts ic
    JOIN speech_vol sv ON ic.gender = sv.gender
    ORDER BY ic.type, ic.gender
""", conn, params={"term_lb": TERM_LOWER_BOUNDARY, "term_ub": TERM_UPPER_BOUNDARY})

df_inj["type_en"] = df_inj["type"].map(TYPE_LABELS)
df_inj["gender_en"] = pd.Categorical(
    df_inj["gender"].map(GENDER_LABELS), categories=["Male", "Female"], ordered=True
)
# Applause and confrontational types tell opposite stories — keep separate
df_inj["category"] = df_inj["type"].map(
    lambda t: "Applause" if t == "Beifall" else "Confrontational & other"
)
type_order = ["Applause", "Heckling", "Dissent", "Laughter", "Amusement"]
df_inj["type_en"] = pd.Categorical(df_inj["type_en"], categories=type_order, ordered=True)
df_inj["per_100k_chars"] = df_inj["per_100k_chars"].astype(float)

p = (
    ggplot(df_inj, aes("type_en", "per_100k_chars", fill="gender_en"))
    + bar_geom_col(position="dodge", width=0.65)
    + facet_wrap("~category", scales="free", nrow=1)
    + bar_scale_fill_manual(values=GENDER_COLORS)
    + scale_y_continuous(labels=lambda l: [f"{v:.1f}" for v in l])
    + labs(
        x=None,
        y="Interjections per 100k chars spoken",
        fill=None,
        title="Interjection rate by type and speaker gender",
    )
    + thesis_theme
    + theme(
        legend_title=element_blank(),
        axis_text_x=element_text(angle=30, ha="right", size=7.5),
    )
)

p.save(f"{FIGURES}/interjection_rate_by_gender.svg", width=TW, height=TW * 0.5, dpi=DPI)
p

### Confrontational interjections by gender within faction

In [ ]:
df_inj_faction = pd.read_sql(f"""
    WITH speech_vol AS (
        SELECT p.gender, {FACTION_MERGE} AS faction,
               SUM(LENGTH(COALESCE(s.speech_content, ''))) AS total_chars
        FROM open_discourse.speeches s
        JOIN open_discourse.politicians p ON s.politician_id = p.id
        JOIN open_discourse.factions f ON s.faction_id = f.id
        WHERE p.gender IN ('männlich', 'weiblich')
          AND f.abbreviation IN ({MAJOR_PARTIES_SQL})
          AND s.electoral_term >= %(term_lb)s
          AND s.electoral_term <= %(term_ub)s
        GROUP BY p.gender, {FACTION_MERGE}
    ),
    inj_counts AS (
        SELECT p.gender, {FACTION_MERGE} AS faction, COUNT(*) AS n
        FROM open_discourse.contributions_extended ce
        JOIN open_discourse.speeches s ON ce.speech_id = s.id
        JOIN open_discourse.politicians p ON s.politician_id = p.id
        JOIN open_discourse.factions f ON s.faction_id = f.id
        WHERE p.gender IN ('männlich', 'weiblich')
          AND ce.type IN ('Zuruf', 'Widerspruch')
          AND f.abbreviation IN ({MAJOR_PARTIES_SQL})
          AND s.electoral_term >= %(term_lb)s
          AND s.electoral_term <= %(term_ub)s
        GROUP BY p.gender, {FACTION_MERGE}
    )
    SELECT ic.gender, ic.faction,
           ic.n::numeric / sv.total_chars * 100000 AS per_100k_chars
    FROM inj_counts ic
    JOIN speech_vol sv ON ic.gender = sv.gender AND ic.faction = sv.faction
    ORDER BY ic.faction, ic.gender
""", conn, params={"term_lb": TERM_LOWER_BOUNDARY, "term_ub": TERM_UPPER_BOUNDARY})

df_inj_faction["faction"] = relabel_factions(df_inj_faction["faction"])
df_inj_faction["gender_en"] = pd.Categorical(
    df_inj_faction["gender"].map(GENDER_LABELS), categories=["Male", "Female"], ordered=True
)
df_inj_faction["per_100k_chars"] = df_inj_faction["per_100k_chars"].astype(float)

faction_order = (
    df_speaker.groupby("faction")["total_chars"].sum()
    .sort_values(ascending=False).index.tolist()
)
df_inj_faction["faction"] = pd.Categorical(
    df_inj_faction["faction"], categories=faction_order, ordered=True
)

p = (
    ggplot(df_inj_faction, aes("gender_en", "per_100k_chars", fill="gender_en"))
    + bar_geom_col(width=0.65)
    + facet_wrap("~faction", nrow=1)
    + bar_scale_fill_manual(values=GENDER_COLORS)
    + scale_y_continuous(labels=lambda l: [f"{v:.1f}" for v in l])
    + labs(
        x=None,
        y="Interjections received per 100k chars",
        title="Confrontational interjections received by speaker gender, within faction",
        caption=f"Includes: Zuruf (heckling) and Widerspruch (dissent). WP{TERM_LOWER_BOUNDARY} – WP{TERM_UPPER_BOUNDARY}.",
    )
    + thesis_theme
    + theme(
        legend_position="none",
        strip_text=element_text(size=7.5),
        axis_text_x=element_text(size=7.5),
        plot_caption=element_text(size=7, colour="#666666", ha="left"),
    )
)

p.save(f"{FIGURES}/heckling_rate_by_gender_faction.svg", width=TW, height=TW * 0.5, dpi=DPI)
p

### Heckling and dissent per party

In [ ]:
# Heckling and dissent by faction (who makes the interjection)
df_heckle = pd.read_sql(f"""
    SELECT
        {FACTION_MERGE} AS faction,
        ce.type,
        COUNT(*) AS n
    FROM open_discourse.contributions_extended ce
    JOIN open_discourse.factions f ON ce.faction_id = f.id
    JOIN open_discourse.speeches s ON ce.speech_id = s.id
    WHERE ce.type IN ('Zuruf', 'Widerspruch')
      AND f.abbreviation IN ({MAJOR_PARTIES_SQL})
      AND s.electoral_term >= %(term_lb)s
      AND s.electoral_term <= %(term_ub)s
    GROUP BY {FACTION_MERGE}, ce.type
    ORDER BY n DESC
""", conn, params={"term_lb": TERM_LOWER_BOUNDARY, "term_ub": TERM_UPPER_BOUNDARY})

df_heckle["faction"] = relabel_factions(df_heckle["faction"])
df_heckle["type_en"] = df_heckle["type"].map(TYPE_LABELS)
df_heckle["type_en"] = pd.Categorical(df_heckle["type_en"], categories=["Heckling", "Dissent"], ordered=True)

order = df_heckle.groupby("faction")["n"].sum().sort_values(ascending=False).index.tolist()
df_heckle["faction"] = pd.Categorical(df_heckle["faction"], categories=order, ordered=True)

p = (
    ggplot(df_heckle, aes("faction", "n", fill="faction"))
    + bar_geom_col(position="stack", width=0.7)
    + bar_scale_fill_manual(values=PARTY_COLORS)
    + scale_y_continuous(labels=lambda l: [f"{v/1000:.0f}k" for v in l])
    + labs(
        x=None,
        y="Total interjections made",
        fill=None,
        title="Heckling and dissent by faction\n (who makes the interjection)",
    )
    + thesis_theme
    + theme(legend_position="none")
)

p.save(f"{FIGURES}/heckling_by_faction.svg", width=TW, height=TW * 0.45, dpi=DPI)
p

## Export stats for Typst

In [ ]:
df_totals = pd.read_sql(f"""
    SELECT
        COUNT(*) AS n_speeches,
        COUNT(DISTINCT s.politician_id) AS n_speakers,
        MIN(s.date)::date AS date_min,
        MAX(s.date)::date AS date_max,
        COUNT(DISTINCT s.electoral_term) AS n_terms
    FROM open_discourse.speeches s
    JOIN open_discourse.politicians p ON s.politician_id = p.id
    JOIN open_discourse.factions f ON s.faction_id = f.id
    WHERE s.electoral_term >= %(term_lb)s
      AND s.electoral_term <= %(term_ub)s
      AND p.gender IN ('männlich', 'weiblich')
      AND f.abbreviation IN ({MAJOR_PARTIES_SQL})
      AND LENGTH(s.speech_content) > 200
""", conn, params={"term_lb": TERM_LOWER_BOUNDARY, "term_ub": TERM_UPPER_BOUNDARY}).iloc[0]

# Female share of total speech volume (chars), pooled across WP12-WP19
_vol = df_terms_gender.groupby("gender")["speeches"].sum()
_female_share = _vol.get("weiblich", 0) / _vol.sum()

# Speaker concentration: top shares of total speech volume
top10_n = max(1, int(len(df_lorenz) * 0.1))
top1_n = max(1, int(len(df_lorenz) * 0.01))
speaker_top10_share = float(df_lorenz.iloc[top10_n - 1]["cum_share"]) / 100
speaker_top1_share = float(df_lorenz.iloc[top1_n - 1]["cum_share"]) / 100
speaker_top10_share_male = float(df_annot.set_index("gender_label").loc["Male", "top10"]) / 100
speaker_top1_share_male = float(df_annot.set_index("gender_label").loc["Male", "top1"]) / 100
speaker_top10_share_female = float(df_annot.set_index("gender_label").loc["Female", "top10"]) / 100
speaker_top1_share_female = float(df_annot.set_index("gender_label").loc["Female", "top1"]) / 100

# Format dates as "D Month YYYY" for use in main text
_fmt_date = lambda d: date.fromisoformat(str(d)).strftime("%-d %B %Y")

stats = {
    "n_speeches":          int(df_totals["n_speeches"]),
    "n_speeches_fmt":      f"{int(df_totals['n_speeches']):,}",
    "n_speakers":          int(df_totals["n_speakers"]),
    "n_speakers_fmt":      f"{int(df_totals['n_speakers']):,}",
    "n_terms":             int(df_totals["n_terms"]),
    "date_min":            _fmt_date(df_totals["date_min"]),
    "date_max":            _fmt_date(df_totals["date_max"]),
    "female_speech_share": round(float(_female_share), 4),
    "female_speech_share_pct": f"{round(float(_female_share) * 100, 1):.1f}",
    "speaker_top10_share": round(speaker_top10_share, 4),
    "speaker_top10_share_pct": f"{round(speaker_top10_share * 100, 1):.1f}",
    "speaker_top1_share": round(speaker_top1_share, 4),
    "speaker_top1_share_pct": f"{round(speaker_top1_share * 100, 1):.1f}",
    "speaker_top10_share_male": round(speaker_top10_share_male, 4),
    "speaker_top10_share_male_pct": f"{round(speaker_top10_share_male * 100, 1):.1f}",
    "speaker_top1_share_male": round(speaker_top1_share_male, 4),
    "speaker_top1_share_male_pct": f"{round(speaker_top1_share_male * 100, 1):.1f}",
    "speaker_top10_share_female": round(speaker_top10_share_female, 4),
    "speaker_top10_share_female_pct": f"{round(speaker_top10_share_female * 100, 1):.1f}",
    "speaker_top1_share_female": round(speaker_top1_share_female, 4),
    "chars_tokens_max_diff_pp": f"{chars_tokens_max_diff_pp:.2f}",
    "speaker_top1_share_female_pct": f"{round(speaker_top1_share_female * 100, 1):.1f}",
}

out_path = Path("../writing/include/vars.json")
out_path.write_text(json.dumps(stats, indent=2, ensure_ascii=False))
print(f"Written to {out_path}")
stats
